# Car-crash detection — YOLOv8s on Roboflow accident-detection

Self-contained Colab notebook that:
1. Downloads the **accident-detection-model** dataset from Roboflow Universe (~3.2k images, YOLO bbox format)
2. Trains **YOLOv8s** for 30 epochs on a free T4
3. Evaluates on the test split (mAP / precision / recall)
4. Runs inference on user-provided accident photos
5. Packages `carcrash.pt` for download

**Runtime:** Free T4 GPU (Runtime → Change runtime type → T4 GPU).
**Time:** ~30–60 min end-to-end (30 epochs).

**You need a free Roboflow API key:** sign up at https://app.roboflow.com → click your avatar → Settings → Roboflow API → copy the Private API Key. Paste it in the cell below when prompted.

Dataset source: [accident-detection-model/accident-detection-model](https://universe.roboflow.com/accident-detection-model/accident-detection-model) (3.2k+ images, YOLO bbox).

## 0. GPU + dependencies

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics==8.3.40 roboflow Pillow

## Mount Google Drive (persists training runs)

Saves every checkpoint straight to Drive, so a Colab disconnect doesn't lose the model. First run pops a permissions dialog — click "Connect to Google Drive". If the runtime dies mid-training, re-mount and rerun the train cell with `resume=True` to continue from the last checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_RUNS = '/content/drive/MyDrive/carcrash_runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
print('Training runs will be saved to:', DRIVE_RUNS)

## 1. Download accident dataset from Roboflow Universe

Paste your Roboflow API key when prompted. It's used once and never stored on disk; if the cell asks you to re-enter it you can rerun safely.

In [ ]:
from getpass import getpass
from pathlib import Path
from roboflow import Roboflow

API_KEY = getpass('Roboflow API key: ').strip()
assert API_KEY, 'Need an API key — see the markdown above for where to find it.'

rf = Roboflow(api_key=API_KEY)
project = rf.workspace('accident-detection-model').project('accident-detection-model')
version = project.version(1)  # version 1 is the original; pick the latest from the UI if newer exists
dataset = version.download('yolov8')

DST = Path(dataset.location)
print('Dataset at:', DST)
print('Top level:', [p.name for p in DST.iterdir()])
print('data.yaml :\n', (DST / 'data.yaml').read_text())

### Tidy data.yaml — fix relative paths and class names if needed

Roboflow writes `train: ../train/images` which is relative to data.yaml; we'll convert to absolute paths to be safe under any working dir.

In [ ]:
import yaml
data_path = DST / 'data.yaml'
data = yaml.safe_load(data_path.read_text())
data['path'] = str(DST)
for split in ('train', 'val', 'test'):
    if split in data and isinstance(data[split], str):
        data[split] = data[split].replace('../', '')
data_path.write_text(yaml.safe_dump(data))
print(data_path.read_text())

### Sanity check: visualise a few labelled accident frames

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import random

NAMES = data['names'] if isinstance(data['names'], list) else list(data['names'].values())
print('Classes:', NAMES)

img_dir = DST / 'train' / 'images'
lbl_dir = DST / 'train' / 'labels'
samples = random.sample(list(img_dir.glob('*'))[:200], k=6)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flat, samples):
    img = Image.open(img_path).convert('RGB')
    W, H = img.size
    draw = ImageDraw.Draw(img)
    lbl_path = lbl_dir / (img_path.stem + '.txt')
    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            parts = line.split()
            if len(parts) < 5: continue
            c, x, y, w, h = int(parts[0]), *map(float, parts[1:5])
            x1, y1 = (x - w/2) * W, (y - h/2) * H
            x2, y2 = (x + w/2) * W, (y + h/2) * H
            draw.rectangle([x1, y1, x2, y2], outline='red', width=3)
            draw.text((x1, max(0, y1 - 12)), NAMES[c], fill='red')
    ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Train YOLOv8s

Same recipe as the fire notebook: AdamW + cosine LR + mosaic for the first 40 epochs.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
results = model.train(
    data=str(DST / 'data.yaml'),
    epochs=30,
    imgsz=640,
    batch=32,
    patience=15,
    close_mosaic=10,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    project=DRIVE_RUNS,
    name='carcrash_yolov8s',
    device=0,
    workers=2,
    seed=42,
    verbose=True,
)

## 3. Evaluate on test split

In [ ]:
from pathlib import Path
best = Path(DRIVE_RUNS) / 'carcrash_yolov8s' / 'weights' / 'best.pt'
model = YOLO(str(best))
metrics = model.val(data=str(DST / 'data.yaml'), split='test', imgsz=640, plots=True)
print('mAP@0.5      :', round(float(metrics.box.map50), 4))
print('mAP@0.5:0.95 :', round(float(metrics.box.map), 4))
print('Precision    :', round(float(metrics.box.mp), 4))
print('Recall       :', round(float(metrics.box.mr), 4))

## 4. Sanity-check on real accident photos

In [ ]:
from pathlib import Path
TEST_DIR = Path('/content/test_crash')
TEST_DIR.mkdir(exist_ok=True)

samples = [
    'https://upload.wikimedia.org/wikipedia/commons/thumb/c/c2/Car_crash_3.jpg/1024px-Car_crash_3.jpg',
    'https://upload.wikimedia.org/wikipedia/commons/thumb/2/24/Car-crash-accident-front-end-damage.jpg/1024px-Car-crash-accident-front-end-damage.jpg',
]
for i, url in enumerate(samples):
    out = TEST_DIR / f'sample_{i}.jpg'
    if not out.exists():
        !wget -q -O {out} {url}

results = model.predict(source=str(TEST_DIR), imgsz=640, conf=0.25, save=True)
for r in results:
    print(r.path, '→', len(r.boxes), 'detections')
    for b in r.boxes:
        print(f'  {model.names[int(b.cls)]} conf={float(b.conf):.2f}')

import glob
from IPython.display import Image as IPyImage, display
for p in sorted(glob.glob(str(Path(results[0].save_dir) / '*.jpg'))):
    print(p); display(IPyImage(p))

## 5. Package for local use

In [ ]:
import shutil
from google.colab import files

src = Path(DRIVE_RUNS) / 'carcrash_yolov8s'
out = Path('/content/carcrash_model_release'); out.mkdir(exist_ok=True)
shutil.copy(src / 'weights/best.pt', out / 'carcrash.pt')
for f in ['results.png', 'results.csv', 'confusion_matrix.png', 'PR_curve.png']:
    if (src / f).exists():
        shutil.copy(src / f, out / f)

# Belt-and-braces: the weights are already on Drive; this just bundles a clean release.
shutil.make_archive('/content/carcrash_model_release', 'zip', out)
files.download('/content/carcrash_model_release.zip')

## Drop the trained weights into the app

Unzip the release and copy `carcrash.pt` to `model/carcrash.pt`. The endpoint `POST /api/v1/detect/carcrash` will start serving on the next `uvicorn` restart.

**Expected metrics** on a clean run with this dataset and these hyperparameters:

| Metric | Expected |
| --- | --- |
| mAP@0.5 | 0.65 – 0.80 |
| Precision | 0.75 – 0.85 |
| Recall | 0.55 – 0.70 |

If recall is below 0.5, try another 20 epochs or step up to `yolov8m.pt`.